In [3]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cpu


### **1-2. Monte Carlo(MC)와 TD Learning의 학습 과정 및 성능 비교**
1. 0번부터 6번까지 총 7개의 상태로 구성된 1차원 환경 구현
- 로봇은 항상 3번 상태에서 시작
- 매번 왼쪽 또는 오른쪽으로 50% 확률로 이동
- 0번에 도착하면 보상 0점을 받고 종료
- 6번에 도착하면 보상 +1점을 받고 종료
- 그 외의 이동에선 보상 0점

2. Monte Carlo 학습
- 에피소드가 종료된 후 학습
- 실제로 얻은 누적 보상 계산
- 학습률 적용해 상태 가치 업데이트

3. TD Learning 구현
- 다음 상태의 추정 가치 이용
- TD Target과 TD Error 계산
- 학습률을 적용해 상태 가치 업데이트

4. 학습 결과 비교
- 에피소드 수: 300
- 학습률: 0.05
- 할인률: 1.0
- 모든 상태 가치의 초기값: 0

5. 학습이 완료되면 아래 내용 출력
- 각 상태의 실제 가치
- MC가 학습한 상태 가치
- TD가 학습한 상태 가치
- 에피소드멸 MC와 TD의 RMSE 변화 그래프

6. 아래 질문에 대답
- MC와 TD Learning은 상태 가치를 업데이트하는 시점과 방법에서 어떤 차이가 있나요?
- RMSE가 작다는것은 무엇을 의미하나요?
- 실습결과에서 MC와 TD중 어느 알고리즘이 더 빠르게 참의 값에 가까워졌고, 그 이유는?

In [4]:
N_STATES = 7
START_STATE = 3
LEFT_END = 0
RIGHT_END = 6

EPISODES = 300
ALPHA = 0.05
GAMMA = 1.0

TRUE_VALUES = np.arange(7) / 5
TRUE_VALUES[6] = 1.0

In [5]:
def step(state, rng):
    # 왼쪽, 오른쪽 50%
    action = rng.choice([-1, 1])
    next_state = state + action

    # 6번 도착 시 +1점
    reward = 1.0 if next_state == RIGHT_END else 0.0

    # 종료: 0번 or 6번 도착
    done = next_state in (LEFT_END, RIGHT_END)

    return next_state, reward, done

In [6]:
# 몬테카를로 구현

def monte_carlo(episodes=300, alpha=0.05, gamma=1.0, seed=2026):
    rng = np.random.default_rng(seed)

    # 모든 상태 가치 0으로 초기화
    values = np.zeros(N_STATES)

    # 에피소드별 학습 결과 저장
    history = []

    for episode in range(episodes):
        state = START_STATE
        trajectory = []

        # 1. 에피소드 실행
        while True:
            next_state, reward, done = step(state, rng)

            trajectory.append((state, reward))
            state = next_state

            if done:
                break

        # 2. 종료 후 누적 보상 계산
        G = 0.0
        visited = set()

        for state, reward in reversed(trajectory):
            G = reward + gamma * G

            # First-visit MC:
            # 에피소드에서 처음 방문한 상태만 업데이트
            if state not in visited:
                visited.add(state)

        visited.clear()

        for state, reward in trajectory:
            if state not in visited:
                visited.add(state)

                # gamma=1.0, 중간 보상=0이므로
                # 모든 방문 상태의 리턴은 최종 보상과 같음
                values[state] += alpha * (G - values[state])

        # 3. RMSE 기록 (1~5번 상태)
        rmse = np.sqrt(
            np.mean((values[1:6] - TRUE_VALUES[1:6]) ** 2)
        )

        history.append(rmse)

    return values, history

In [7]:
mc_values, mc_history = monte_carlo(
    episodes=EPISODES,
    alpha=ALPHA,
    gamma=GAMMA
)

print("MC 학습 결과:", np.round(mc_values, 3))
print("최종 RMSE:", round(mc_history[-1], 4))

MC 학습 결과: [0.    0.252 0.43  0.61  0.794 0.89  0.   ]
최종 RMSE: 0.0563
